# Prescient LMP Analysis With Hydro (Q1)

This notebook processes **raw CRC simulation outputs** and benchmarks them against:

1. Hydro `btheta` UC+ED run (`results_btheta`)
2. Hydro `btheta` UC-only run (`results_btheta_uc_only`)
3. No-hydro 365-day benchmark run, sliced to Q1 only

The goal is to evaluate whether PCM behavior is converging toward the paper benchmark while keeping Q1 as a low-cost proxy for full-year simulation.

## 1. Paths, Scenarios, and Simulation Configuration Notes

### Hydro simulations used here

- `submit_job_btheta.py` -> `run_coal_prescient_btheta.py` -> `Prescient_2/results_btheta`
- `submit_job_btheta_uc_only.py` -> `run_coal_prescient_btheta_uc_only.py` -> `Prescient_2/results_btheta_uc_only`

### Config differences (from run scripts)

| Parameter | Hydro Btheta UC+ED | Hydro Btheta UC-only |
|---|---:|---:|
| `simulate_out_of_sample` | `True` | `False` |
| `run_sced_with_persistent_forecast_errors` | `True` | `False` |
| `sced_horizon` | `6` | `1` |
| `ruc_horizon` | `36` | `36` |
| `price_threshold` | `1000` | `1000` |
| Network | `btheta` | `btheta` |
| Intended run horizon | 90 days | 90 days |

Q1 benchmarking is intentional to reduce computational cost while preserving seasonal signal.

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

repo_root = Path('/Users/yilu/Documents/GitHub/idaes-gtep')
run_root = repo_root / 'gtep' / 'data' / 'retirement_allowed_no_extreme_half_load'
pcm_dir = repo_root / 'gtep' / 'pcm_analysis'

paths = {
    'Hydro Btheta UC+ED': run_root / 'Prescient_2' / 'results_btheta',
    'Hydro Btheta UC-only': run_root / 'Prescient_2' / 'results_btheta_uc_only',
    'No-hydro Benchmark Q1': Path('/Users/yilu/Documents/development/nd/research/gtep/123_bus_coal/results/retirement_allowed_no_extreme/12mon_no_hydro_with_curtailment'),
}
colors = {
    'Hydro Btheta UC+ED': 'steelblue',
    'Hydro Btheta UC-only': 'darkorange',
    'No-hydro Benchmark Q1': 'forestgreen',
}

paper_ref = {
    'typical_lmp_range': '10-50 $/MWh (mostly positive)',
    'q2_daily_dlr_mean_lmp': 18.66,
    'q2_hourly_dlr_mean_lmp': 17.98,
    'q2_daily_operational_cost_musd': 8.09,
}

for k, v in paths.items():
    print(f'{k:24s} -> {v}')
print('\nOutput dir:', pcm_dir)

## 2. CRC Raw Log Context (latest hydro logs)

This section reads the latest two hydro `.o*` logs to document run context in the benchmarking notebook.

In [ ]:
def latest_log(prefix: str):
    logs = sorted(run_root.glob(prefix), key=lambda p: p.stat().st_mtime, reverse=True)
    return logs[0] if logs else None

log_files = {
    'Hydro Btheta UC+ED': latest_log('ERCOT_btheta_hydro.o*'),
    'Hydro Btheta UC-only': latest_log('ERCOT_btheta_uc_hydro.o*'),
}

rows = []
for scenario, p in log_files.items():
    if p is None:
        rows.append({'Scenario': scenario, 'Log': None, 'HasTraceback': None, 'SizeBytes': None})
        continue
    txt = p.read_text(errors='ignore')
    rows.append({
        'Scenario': scenario,
        'Log': str(p),
        'HasTraceback': ('Traceback' in txt) or ('TypeError' in txt) or ('ValueError' in txt),
        'SizeBytes': p.stat().st_size,
    })

log_df = pd.DataFrame(rows)
display(log_df)

for _, r in log_df.dropna(subset=['Log']).iterrows():
    txt = Path(r['Log']).read_text(errors='ignore').splitlines()
    print('\n' + '='*90)
    print(r['Scenario'])
    print('HEAD:')
    print('\n'.join(txt[:6]))
    print('TAIL:')
    print('\n'.join(txt[-10:]))

## 3. Raw Data Load Helpers

In [ ]:
def safe_read_csv(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()
    try:
        return pd.read_csv(path)
    except pd.errors.EmptyDataError:
        return pd.DataFrame()

def to_dt(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    if 'Date' in df.columns and 'Hour' in df.columns:
        if 'Minute' not in df.columns:
            df['Minute'] = 0
        df['Datetime'] = (
            pd.to_datetime(df['Date'], errors='coerce')
            + pd.to_timedelta(df['Hour'], unit='h')
            + pd.to_timedelta(df['Minute'], unit='m')
        )
    return df

def load_raw_results(root: Path) -> dict:
    data = {
        'bus': to_dt(safe_read_csv(root / 'bus_detail.csv')),
        'renew': to_dt(safe_read_csv(root / 'renewables_detail.csv')),
        'thermal': to_dt(safe_read_csv(root / 'thermal_detail.csv')),
        'hourly': to_dt(safe_read_csv(root / 'hourly_summary.csv')),
        'daily': safe_read_csv(root / 'daily_summary.csv'),
    }
    return data

def q1_only(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty or 'Datetime' not in df.columns:
        return df
    return df[df['Datetime'].dt.month <= 3].copy()

raw = {}
for name, p in paths.items():
    d = load_raw_results(p)
    for k in ['bus', 'renew', 'thermal', 'hourly']:
        d[k] = q1_only(d[k])
    raw[name] = d

availability = []
for name, d in raw.items():
    for key in ['bus', 'renew', 'thermal', 'hourly']:
        df = d[key]
        if not df.empty and 'Datetime' in df.columns:
            s, e = df['Datetime'].min(), df['Datetime'].max()
        else:
            s, e = pd.NaT, pd.NaT
        availability.append({'Scenario': name, 'Dataset': key, 'Rows': len(df), 'Start': s, 'End': e})

availability_df = pd.DataFrame(availability)
display(availability_df.sort_values(['Scenario', 'Dataset']))

## 4. Common Q1 Comparison Window and Export Analysis Files

To avoid biased comparison when data windows differ, we benchmark on the overlap window across all scenario bus-level timestamps.

In [ ]:
ranges = []
for name, d in raw.items():
    b = d['bus']
    if not b.empty and 'Datetime' in b.columns:
        ranges.append((name, b['Datetime'].min(), b['Datetime'].max()))

if len(ranges) < 2:
    raise RuntimeError('Not enough valid datasets for overlap benchmarking.')

start = max(r[1] for r in ranges)
end = min(r[2] for r in ranges)
if start > end:
    raise RuntimeError('No overlap window across scenarios.')

print('Overlap window:', start, '->', end)
for r in ranges:
    print(f'{r[0]:24s}: {r[1]} -> {r[2]}')

windowed = {}
for name, d in raw.items():
    windowed[name] = {}
    for k, df in d.items():
        if isinstance(df, pd.DataFrame) and (not df.empty) and ('Datetime' in df.columns):
            windowed[name][k] = df[(df['Datetime'] >= start) & (df['Datetime'] <= end)].copy()
        else:
            windowed[name][k] = df

def wide_lmp(bus_df):
    if bus_df.empty:
        return pd.DataFrame()
    lmp_col = 'LMP DA' if 'LMP DA' in bus_df.columns else 'LMP'
    piv = bus_df.pivot_table(index='Datetime', columns='Bus', values=lmp_col, aggfunc='mean')
    piv.columns = [f'{c}_LMP' for c in piv.columns]
    return piv.sort_index()

def wide_dispatch(thermal_df, renew_df):
    out = []
    if (not thermal_df.empty) and {'Datetime','Generator','Dispatch'}.issubset(thermal_df.columns):
        t = thermal_df.pivot_table(index='Datetime', columns='Generator', values='Dispatch', aggfunc='mean')
        t.columns = [f'Gen{c}_ThermalDispatch' for c in t.columns]
        out.append(t)
    if (not renew_df.empty) and {'Datetime','Generator','Output'}.issubset(renew_df.columns):
        r = renew_df.pivot_table(index='Datetime', columns='Generator', values='Output', aggfunc='mean')
        r.columns = [f'Gen{c}_RenewOutput' for c in r.columns]
        out.append(r)
    if not out:
        return pd.DataFrame()
    z = out[0]
    for part in out[1:]:
        z = z.join(part, how='outer')
    return z.sort_index()

out_names = {
    'Hydro Btheta UC+ED': ('Bus_LMP_hydro_btheta.csv', 'Generator_Dispatch_hydro_btheta.csv'),
    'Hydro Btheta UC-only': ('Bus_LMP_hydro_uc_only.csv', 'Generator_Dispatch_hydro_uc_only.csv'),
    'No-hydro Benchmark Q1': ('Bus_LMP_no_hydro_q1.csv', 'Generator_Dispatch_no_hydro_q1.csv'),
}

for name, (lmp_name, disp_name) in out_names.items():
    b = windowed[name]['bus']
    t = windowed[name]['thermal']
    r = windowed[name]['renew']
    lmp_w = wide_lmp(b)
    disp_w = wide_dispatch(t, r)
    lmp_w.to_csv(pcm_dir / lmp_name)
    disp_w.to_csv(pcm_dir / disp_name)
    print(f'{name}: wrote {lmp_name} {lmp_w.shape}, {disp_name} {disp_w.shape}')

## 5. Quantitative Benchmark Metrics and Plots

In [ ]:
def weighted_lmp(bus_df):
    if bus_df.empty:
        return pd.Series(dtype=float)
    lmp_col = 'LMP DA' if 'LMP DA' in bus_df.columns else 'LMP'
    g = bus_df.groupby('Datetime')
    wsum = g.apply(lambda x: (x['Demand'] * x[lmp_col]).sum())
    dsum = g['Demand'].sum().replace(0, np.nan)
    return (wsum / dsum).fillna(0.0)

metrics = []
for name, d in windowed.items():
    b = d['bus']
    r = d['renew']
    h = d['hourly']
    lmp_col = 'LMP DA' if (not b.empty and 'LMP DA' in b.columns) else 'LMP'
    row = {'Scenario': name}
    if not b.empty:
        row.update({
            'rows_bus': len(b),
            'lmp_mean': float(b[lmp_col].mean()),
            'lmp_median': float(b[lmp_col].median()),
            'lmp_min': float(b[lmp_col].min()),
            'lmp_max': float(b[lmp_col].max()),
            'neg_lmp_frac': float((b[lmp_col] < 0).mean()),
            'floor_hit_frac': float((b[lmp_col] <= -1000).mean()),
            'lmp_weighted': float((b['Demand'] * b[lmp_col]).sum() / b['Demand'].sum()) if b['Demand'].sum() > 0 else np.nan,
        })
    else:
        row.update({'rows_bus': 0, 'lmp_mean': np.nan, 'lmp_median': np.nan, 'lmp_min': np.nan, 'lmp_max': np.nan, 'neg_lmp_frac': np.nan, 'floor_hit_frac': np.nan, 'lmp_weighted': np.nan})
    row['curtailment_total'] = float(r['Curtailment'].sum()) if (not r.empty and 'Curtailment' in r.columns) else np.nan
    row['overgen_total'] = float(h['OverGeneration'].sum()) if (not h.empty and 'OverGeneration' in h.columns) else np.nan
    row['load_shedding_total'] = float(h['LoadShedding'].sum()) if (not h.empty and 'LoadShedding' in h.columns) else np.nan
    metrics.append(row)

metrics_df = pd.DataFrame(metrics)
display(metrics_df)

summary = {
    'overlap_start': str(start),
    'overlap_end': str(end),
    'paper_ref': paper_ref,
    'metrics': metrics_df.to_dict(orient='records'),
}
(pcm_dir / 'PCM_result_hydro_q1_comparison.json').write_text(json.dumps(summary, indent=2))
print('Wrote PCM_result_hydro_q1_comparison.json')

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

ax = axes[0, 0]
for name, d in windowed.items():
    s = weighted_lmp(d['bus'])
    if not s.empty:
        ax.plot(s.index, s.values, label=name, color=colors[name], linewidth=1.0)
ax.axhline(0, color='black', linewidth=0.7)
ax.set_title('Hourly Load-Weighted LMP')
ax.legend(fontsize=8)

ax = axes[0, 1]
for name, d in windowed.items():
    s = weighted_lmp(d['bus'])
    if not s.empty:
        daily = s.resample('D').mean()
        ax.plot(daily.index, daily.values, marker='o', label=name, color=colors[name])
ax.axhline(0, color='black', linewidth=0.7)
ax.set_title('Daily Load-Weighted LMP')
ax.legend(fontsize=8)

ax = axes[1, 0]
for name, d in windowed.items():
    b = d['bus']
    if not b.empty:
        c = 'LMP DA' if 'LMP DA' in b.columns else 'LMP'
        ax.hist(b[c], bins=120, density=True, alpha=0.35, label=name, color=colors[name])
ax.axvline(0, color='black', linewidth=0.7)
ax.set_title('LMP Distribution')
ax.legend(fontsize=8)

ax = axes[1, 1]
for name, d in windowed.items():
    b = d['bus']
    if not b.empty:
        c = 'LMP DA' if 'LMP DA' in b.columns else 'LMP'
        tmp = b.copy()
        tmp['H'] = tmp['Datetime'].dt.hour
        neg = tmp.groupby('H').apply(lambda x: (x[c] < 0).mean() * 100)
        ax.plot(neg.index, neg.values, marker='o', label=name, color=colors[name])
ax.set_title('Negative LMP Frequency by Hour')
ax.set_xlabel('Hour')
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## 6. Paper Benchmark Comparison Table (hardcoded references from existing analysis)

Paper references reused from existing `prescient_lmp_analysis.ipynb` benchmark section:

- Typical LMP behavior: **10-50 $/MWh, mostly positive**
- Q2 Daily DLR mean LMP: **18.66 $/MWh**
- Q2 Hourly DLR mean LMP: **17.98 $/MWh**
- Q2 normal-day operational cost (Table VII): **$8.09M/day**

In [ ]:
paper_table = pd.DataFrame([
    {'Metric': 'Typical LMP range', 'Paper': paper_ref['typical_lmp_range']},
    {'Metric': 'Q2 Daily DLR mean LMP', 'Paper': paper_ref['q2_daily_dlr_mean_lmp']},
    {'Metric': 'Q2 Hourly DLR mean LMP', 'Paper': paper_ref['q2_hourly_dlr_mean_lmp']},
    {'Metric': 'Q2 daily operational cost ($M/day)', 'Paper': paper_ref['q2_daily_operational_cost_musd']},
])

display(paper_table)
display(metrics_df[['Scenario','lmp_mean','lmp_median','lmp_min','lmp_max','neg_lmp_frac','lmp_weighted','curtailment_total']].sort_values('Scenario'))

## 7. Summary Discussion

### What this notebook produced

- Processed raw CRC hydro outputs from `results_btheta` and `results_btheta_uc_only`.
- Benchmarked both against no-hydro Q1 data from the 365-day run.
- Exported analysis-ready wide files and a JSON summary in `gtep/pcm_analysis`.

### How to interpret results for model tuning

1. If hydro runs still have high negative-LMP fraction and floor hits, PCM remains materially different from paper's mostly positive-price behavior.
2. Compare hydro UC+ED vs hydro UC-only first to isolate dispatch-mode impact under same network.
3. Compare both hydro cases against no-hydro Q1 to quantify how hydro/case setup shifts prices and curtailment.
4. Use this Q1 benchmark loop for fast iteration; once behavior is closer to paper references, expand horizon for robustness checks.

### Notes

- Q1 is intentionally used as a computationally cheaper benchmark proxy.
- Paper references here are hardcoded from prior notebook benchmark sections, not extracted directly from PDF.